In [1]:
from datasets import load_dataset

csv_path = "/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Testing/claritygraph.csv"

dataset = load_dataset(
        "csv", data_files=[csv_path], split="train", delimiter=",", column_names=["title", "text"]
    )

/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
)

dpr_ctx_encoder_model_name = "facebook/dpr-ctx_encoder-multiset-base"

dpr_tokenizer = DPRContextEncoderTokenizer.from_pretrained(dpr_ctx_encoder_model_name)
dpr_encoder = DPRContextEncoder.from_pretrained(dpr_ctx_encoder_model_name)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-multiset-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

' input_ids = tokenizer("Hello, is my dog cute ?", return_tensors="pt")["input_ids"]\nembeddings = encoder(input_ids, return_dict=True).pooler_output\nprint(embeddings) '

In [4]:
import numpy as np

def testembed(docs: dict, encoder: DPRContextEncoder = dpr_encoder, tokenizer: DPRContextEncoderTokenizer = dpr_tokenizer):
    input_ids = tokenizer(docs['title'], docs['text'], return_tensors='pt')['input_ids']
    embeddings = encoder(input_ids, return_dict = True).pooler_output
    return {'embeddings': embeddings[0]}

In [5]:
from functools import partial

embedded = dataset.map(partial(testembed))

In [7]:
import os

context_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/contexts'
ver = 0
passages_path = os.path.join(context_dir, f"my_knowledge_dataset_{ver}")

embedded.save_to_disk(passages_path)

Saving the dataset (1/1 shards): 100%|██████████| 99/99 [00:00<00:00, 46697.72 examples/s]


In [8]:
# from datasets import load_from_disk
# dataset = load_from_disk(passages_path)

In [9]:
import faiss

index = faiss.IndexHNSWFlat(768, 128, faiss.METRIC_INNER_PRODUCT)
embedded.add_faiss_index("embeddings", custom_index=index)

100%|██████████| 1/1 [00:00<00:00, 200.28it/s]


Dataset({
    features: ['title', 'text', 'embeddings'],
    num_rows: 99
})

In [10]:
index_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/indices'
index_path = os.path.join(index_dir, f"my_knowledge_dataset_{ver}_hnsw_index.faiss")

embedded.get_index("embeddings").save(index_path)

In [11]:
# dataset.load_faiss_index("embeddings", index_path)

In [12]:
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

rag_name = 'facebook/rag-sequence-nq'

retriever = RagRetriever.from_pretrained(
    rag_name, index_name="custom", indexed_dataset=embedded
)
rag_model = RagSequenceForGeneration.from_pretrained(rag_name, retriever=retriever)
rag_tokenizer = RagTokenizer.from_pretrained(rag_name)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fr

In [14]:
question = 'Who is the parent of Robert?'
input_ids = rag_tokenizer.question_encoder(question, return_tensors='pt')['input_ids']
generated = rag_model.generate(input_ids)

: 